## Sabemos que hay una cantidad significativa de duplicados, nulos e inconsistencias en tm_envios, asi que empezaremos por ahí.

In [40]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

envios = spark.read.table("bronze.bronze_tms_envios")
print("Registros iniciales:", envios.count())
duplicados = envios.count() - envios.dropDuplicates(["id_envio"]).count()
print(f"Duplicados encontrados: {duplicados}")
envios = envios.dropDuplicates(["id_envio"])
print(f"Registros después de eliminar duplicados: {envios.count()}")



display(envios.select([count(when(col(c).isNull(), c)).alias(c) for c in envios.columns]))
#relleneamos nulos con no aplica si es el caso, pues los NULLs en estas entradas indican que no se entregó el paquete en N intentos.
envios = envios.fillna({
    "resultado_intento1": "NO APLICA",
    "resultado_intento2": "NO APLICA",
    "motivo_fallo_cod": "NO APLICA"
})
print("rellenando nulos...")

fechas_invalidas = envios.filter(
    F.col("fec_entrega_real").isNotNull() &
    (F.col("fec_entrega_real") < F.col("fec_recepcion"))
)

print("Fechas inválidas:", fechas_invalidas.count())

fechas_invalidas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("errors.error_fechas")

# Eliminamos solo las inconsistentes
envios = envios.filter(
    F.col("fec_entrega_real").isNull() |
    (F.col("fec_entrega_real") >= F.col("fec_recepcion"))
)

pesos_invalidos = envios.filter(
    (F.col("peso_kg") <= 0) |
    (F.col("peso_kg") > 100)
)

print("Pesos inválidos:", pesos_invalidos.count())

pesos_invalidos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("errors.error_pesos")

envios = envios.filter(
    (F.col("peso_kg") > 0) &
    (F.col("peso_kg") <= 100)
)

errores_fechas = fechas_invalidas.withColumn(
    "motivo_error",
    F.lit("Fecha de entrega anterior a la recepción")
)

errores_pesos = pesos_invalidos.withColumn(
    "motivo_error",
    F.lit("Peso fuera del rango permitido")
)

errores = errores_fechas.unionByName(errores_pesos)

errores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("errors.errores_calidad")


envios.printSchema()

print(envios.count())
display(envios.limit(10))

print("limpieza exitosa")

total = 2020000

print(f"Duplicados : {duplicados/total:.2%}")
print(f"Fechas     : {fechas_invalidas.count()/total:.2%}")
print(f"Pesos      : {pesos_invalidos.count()/total:.2%}")
print(f"Conformes  : {envios.count()/total:.2%}")

StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 42, Finished, Available, Finished, False)

Registros iniciales: 2020000
Duplicados encontrados: 20201
Registros después de eliminar duplicados: 1999799


SynapseWidget(Synapse.DataFrame, d5799c84-ec75-4fd0-8fc2-7e3c1df6a7cc)

rellenando nulos...
Fechas inválidas: 4981
Pesos inválidos: 9979
root
 |-- id_envio: long (nullable = true)
 |-- id_remitente: long (nullable = true)
 |-- cond_id: long (nullable = true)
 |-- id_zona_destino: integer (nullable = true)
 |-- tip_paquete: string (nullable = true)
 |-- peso_kg: double (nullable = true)
 |-- fec_recepcion: date (nullable = true)
 |-- hra_recepcion: timestamp (nullable = true)
 |-- fec_entrega_programada: date (nullable = true)
 |-- fec_intento1: date (nullable = true)
 |-- hra_intento1: timestamp (nullable = true)
 |-- resultado_intento1: string (nullable = false)
 |-- fec_intento2: date (nullable = true)
 |-- hra_intento2: timestamp (nullable = true)
 |-- resultado_intento2: string (nullable = false)
 |-- fec_entrega_real: date (nullable = true)
 |-- estado_final: string (nullable = true)
 |-- motivo_fallo_cod: string (nullable = false)
 |-- vr_declarado: double (nullable = true)
 |-- Marca_tiempo_ingesta: timestamp (nullable = true)
 |-- sistema_fuente: s

SynapseWidget(Synapse.DataFrame, 06f307e8-c3e4-46a1-9a23-e81d3428861a)

limpieza exitosa
Duplicados : 1.00%
Fechas     : 0.25%
Pesos      : 0.49%
Conformes  : 98.26%


### Se hace el mismo proceso de verificación de duplicados, relleno de nulos, reporte de partes porcentuales para cada una de las deltatables restantes

In [41]:
from pyspark.sql.functions import col, count, when

destinatarios = spark.read.table("bronze.bronze_cal_destinatarios")
print("Registros iniciales:", destinatarios.count())
duplicados = destinatarios.count() - destinatarios.dropDuplicates(["id_calificacion"]).count()
print(f"Duplicados encontrados: {duplicados}")
destinatarios = destinatarios.dropDuplicates(["id_calificacion"])
print(f"Registros después de eliminar duplicados: {destinatarios.count()}")


# Cuenta los nulos por cada columna
display(destinatarios.select([count(when(col(c).isNull(), c)).alias(c) for c in destinatarios.columns]))
dest_nulos = destinatarios.filter(F.col("comentario_texto").isNull())

print("rellenando nulos...")
destinatarios = destinatarios.fillna({"comentario_texto": "Ninguno"})

display(destinatarios.limit(10))
print("limpieza exitosa")

total = 300000

print(f"Duplicados : {duplicados/total:.4%}")
print(f"Nulos      : { dest_nulos.count()/total:.4}")
print(f"Conformes  : {destinatarios.count()/total:.4%}")


StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 43, Finished, Available, Finished, False)

Registros iniciales: 300000
Duplicados encontrados: 4
Registros después de eliminar duplicados: 299996


SynapseWidget(Synapse.DataFrame, d06c10ae-60a7-4a02-b79c-c7c779e086a3)

rellenando nulos...


SynapseWidget(Synapse.DataFrame, 0b006877-6544-421b-8742-2eccf5f5b964)

limpieza exitosa
Duplicados : 0.0013%
Nulos      : 0.05054
Conformes  : 99.9987%


### Hay algunas tablas como esta que se generó sin duplicados y en algunos casos sin nulos también

In [42]:
remitentes = spark.read.table("bronze.bronze_cli_remitentes")
print("Registros iniciales:", remitentes.count())
duplicados = remitentes.count() - remitentes.dropDuplicates(["id_remitente"]).count()
print(f"Duplicados encontrados: {duplicados}")
remitentes = remitentes.dropDuplicates(["id_remitente"])
print(f"Registros después de eliminar duplicados: {remitentes.count()}")

# Cuenta los nulos por cada columna
display(remitentes.select([count(when(col(c).isNull(), c)).alias(c) for c in remitentes.columns]))

print("No hay nulos por rellenar")

display(remitentes.limit(10))
print("limpieza exitosa")

total = 200

print(f"Duplicados : {duplicados/total:.2%}")
print(f"Conformes  : {remitentes.count()/total:.2%}")

StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 44, Finished, Available, Finished, False)

Registros iniciales: 200
Duplicados encontrados: 0
Registros después de eliminar duplicados: 200


SynapseWidget(Synapse.DataFrame, c55fbe33-3fd2-4ee7-aae4-3cdde1cbac78)

No hay nulos por rellenar


SynapseWidget(Synapse.DataFrame, 22b15228-9486-4f88-bff3-54576e13d81e)

limpieza exitosa
Duplicados : 0.00%
Conformes  : 100.00%


#### tabla de novedades

In [43]:
novedades = spark.read.table("bronze.bronze_dir_novedades")
print("Registros iniciales:", novedades.count())
duplicados = novedades.count() - novedades.dropDuplicates(["id_novedad"]).count()
print(f"Duplicados encontrados: {duplicados}")
novedades = novedades.dropDuplicates(["id_novedad"])
print(f"Registros después de eliminar duplicados: {novedades.count()}")

# Cuenta los nulos por cada columna
display(novedades.select([count(when(col(c).isNull(), c)).alias(c) for c in novedades.columns]))
print("No hay nulos por rellenar")


display(novedades.limit(10))
print("limpieza exitosa")

total = 150000

print(f"Duplicados : {duplicados/total:.3%}")
print(f"Conformes  : {novedades.count()/total:.3%}")

StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 45, Finished, Available, Finished, False)

Registros iniciales: 150000
Duplicados encontrados: 2
Registros después de eliminar duplicados: 149998


SynapseWidget(Synapse.DataFrame, 78277c83-de41-440b-91a3-5d0ecf4d1717)

No hay nulos por rellenar


SynapseWidget(Synapse.DataFrame, f6cff497-e3f1-466a-a576-ca4ecf5df265)

limpieza exitosa
Duplicados : 0.001%
Conformes  : 99.999%


#### tabla de novedades

In [44]:
geo = spark.read.table("bronze.bronze_geo_zonas")
print("Registros iniciales:", geo.count())
duplicados = geo.count() - geo.dropDuplicates(["id_zona"]).count()
print(f"Duplicados encontrados: {duplicados}")
geo = geo.dropDuplicates(["id_zona"])

print(f"Registros después de eliminar duplicados: {geo.count()}")

# Cuenta los nulos por cada columna
display(geo.select([count(when(col(c).isNull(), c)).alias(c) for c in geo.columns]))
print("No hay nulos por rellenar")


display(geo.limit(10))
print("limpieza exitosa")

total = 300

print(f"Duplicados : {duplicados/total:.2%}")
print(f"Conformes  : {geo.count()/total:.2%}")

StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 46, Finished, Available, Finished, False)

Registros iniciales: 300
Duplicados encontrados: 107
Registros después de eliminar duplicados: 193


SynapseWidget(Synapse.DataFrame, bb5ba7ef-c34d-4234-8698-67ae955389c8)

No hay nulos por rellenar


SynapseWidget(Synapse.DataFrame, c2f6bcfe-cbb9-422f-973c-d4f12e4cea31)

limpieza exitosa
Duplicados : 35.67%
Conformes  : 64.33%


#### gps_rutas

In [45]:
ruta = spark.read.table("bronze.bronze_gps_rutas")
print("Registros iniciales:", ruta.count())
duplicados = ruta.count() - ruta.dropDuplicates(["id_ruta"]).count()
print(f"Duplicados encontrados: {duplicados}")
ruta = ruta.dropDuplicates(["id_ruta"])

print(f"Registros después de eliminar duplicados: {ruta.count()}")

# Cuenta los nulos por cada columna
display(ruta.select([count(when(col(c).isNull(), c)).alias(c) for c in ruta.columns]))
print("No hay nulos por rellenar")


display(ruta.limit(10))
print("limpieza exitosa")

total = 100000

print(f"Duplicados : {duplicados/total:.3%}")
print(f"Conformes  : {ruta.count()/total:.3%}")

StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 47, Finished, Available, Finished, False)

Registros iniciales: 100000
Duplicados encontrados: 1
Registros después de eliminar duplicados: 99999


SynapseWidget(Synapse.DataFrame, 24ef2c9a-a3eb-416b-9b18-9c86e97afdd4)

No hay nulos por rellenar


SynapseWidget(Synapse.DataFrame, 73c14847-a09c-4dbe-a8d4-77e0d8a4316f)

limpieza exitosa
Duplicados : 0.001%
Conformes  : 99.999%


In [46]:
cond = spark.read.table("bronze.bronze_ope_conductores")
print("Registros iniciales:", cond.count())
duplicados = cond.count() - cond.dropDuplicates(["cond_id"]).count()
print(f"Duplicados encontrados: {duplicados}")
cond = cond.dropDuplicates(["cond_id"])

print(f"Registros después de eliminar duplicados: {cond.count()}")

# Cuenta los nulos por cada columna
display(cond.select([count(when(col(c).isNull(), c)).alias(c) for c in cond.columns]))
print("No hay nulos por rellenar")


display(cond.limit(10))
print("limpieza exitosa")

total = 500

print(f"Duplicados : {duplicados/total:.2%}")
print(f"Conformes  : {cond.count()/total:.2%}")

StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 48, Finished, Available, Finished, False)

Registros iniciales: 500
Duplicados encontrados: 0
Registros después de eliminar duplicados: 500


SynapseWidget(Synapse.DataFrame, 13ffd16e-105b-4b9a-aaff-679a9abb876a)

No hay nulos por rellenar


SynapseWidget(Synapse.DataFrame, a2da5086-ca9c-4949-9ab9-b6595afac875)

limpieza exitosa
Duplicados : 0.00%
Conformes  : 100.00%


### En este último bloque de codigo enviamos todas las tablas de manera simultanea al schema de silver para seguir a la siguiente fase de transformación

In [47]:

destinatarios.write.format("delta").mode("overwrite").saveAsTable("silver.silver_cal_destinatarios")
print("Enviando Destinatarios a silver")
remitentes.write.format("delta").mode("overwrite").saveAsTable("silver.silver_cli_remitentes")
print("Enviando remitentes a silver")
novedades.write.format("delta").mode("overwrite").saveAsTable("silver.silver_dir_novedades")
print("Enviando novedades a silver")
geo.write.format("delta").mode("overwrite").saveAsTable("silver.silver_geo_zonas")
print("Enviando geo zonas a silver")
ruta.write.format("delta").mode("overwrite").saveAsTable("silver.silver_gps_rutas")
print("Enviando gps rutas a silver")
cond.write.format("delta").mode("overwrite").saveAsTable("silver.silver_ope_conductores")
print("Enviando tms envios a silver")
envios.write.format("delta").mode("overwrite").saveAsTable("silver.silver_tms_envios")


StatementMeta(, 240749bf-ad78-4334-819a-4c6cf290e1a3, 49, Finished, Available, Finished, False)

Enviando Destinatarios a silver
Enviando remitentes a silver
Enviando novedades a silver
Enviando geo zonas a silver
Enviando gps rutas a silver
Enviando tms envios a silver
